# 05 — Goal 4: perceptual family alignment

Separates the distributed four-RA main annotations from the crossed 70-file reliability subset, evaluates family alignment, and compares both with merged broad 2RA labels.

Every displayed denominator and paper-facing visual is also saved under separate `figures/` and `tables/` folders within `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed. The final cell explains whether the next stage is allowed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def read_optional_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    if stem.with_suffix(".parquet").exists() or stem.with_suffix(".csv").exists():
        return read_stage(relative_without_suffix)
    print("OPTIONAL TABLE NOT AVAILABLE:", relative_without_suffix)
    return pd.DataFrame()

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder / "tables"
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder / "figures"
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)
print("Visualization outputs:", VIZ_ROOT)


Primary alignment excludes competing speech and non-task content because the estimand is family perceptual alignment, not source recognition. The broad metadata direction gate must be confirmed from the RA codebook before the 2RA comparison runs. The main annotation set has one independent RA per recording; agreement and four-RA consensus are estimated only in `Reliability/<RA name>/`, where the same files were rated by all four RAs.

In [ ]:
RUN_GOAL_4 = False
if RUN_GOAL_4:
    run_cli("human-qc", "--schema", "config/human_qc_schema.yaml")
else:
    print("Using existing Goal 4 outputs.")


In [ ]:
main_coverage = read_stage("04_analysis/human_qc/main_distributed_item_coverage")
main_design = read_stage("04_analysis/human_qc/main_distributed_design_summary")
main_ratings = read_stage("04_analysis/human_qc/main_distributed_ratings_long")
main_workload = read_stage("04_analysis/human_qc/main_rater_workload_and_prevalence")
reliability_status = read_stage("04_analysis/human_qc/reliability_analysis_status")
reliability_coverage = read_optional_stage("04_analysis/human_qc/reliability_item_coverage")
reliability_ratings = read_optional_stage("04_analysis/human_qc/reliability_ratings_long")
agreement = read_optional_stage("04_analysis/human_qc/reliability_interrater_agreement_complete")
consensus = read_optional_stage("04_analysis/human_qc/reliability_four_ra_consensus_primary")
direction_audit = read_stage("04_analysis/human_qc/two_ra_broad_direction_and_scale_audit")

display(direction_audit)
save_table(direction_audit, "05_goal4", "direction_and_scale_audit")
save_table(main_design, "05_goal4", "main_distributed_design_summary")
save_table(main_workload, "05_goal4", "main_rater_workload_and_prevalence")
save_table(reliability_status, "05_goal4", "reliability_analysis_status")
display(main_design)
display(main_workload)
display(reliability_status)


In [ ]:
# Main-set coverage: every recording-family should have exactly one RA.
main_matrix = (
    main_ratings.assign(rated=1)
    .pivot_table(index=["file_name", "category"], columns="rater_id", values="rated", aggfunc="max", fill_value=0)
)
save_table(main_matrix.reset_index(), "05_goal4", "main_distributed_coverage_matrix")
fig, ax = plt.subplots(figsize=(10, min(18, max(5, .08 * len(main_matrix)))))
sns.heatmap(main_matrix, cmap=["#f2f2f2", "#4C78A8"], cbar=False, ax=ax)
ax.set(title="Main distributed coverage: one RA per item", xlabel="Rater", ylabel="Recording × perceptual family")
ax.tick_params(axis="y", labelleft=False)
save_figure(fig, "05_goal4", "main_distributed_rating_coverage")
plt.show()

# Reliability coverage: every recording-family should have all four RAs.
if not reliability_ratings.empty:
    reliability_matrix = (
        reliability_ratings.assign(rated=1)
        .pivot_table(index=["file_name", "category"], columns="rater_id", values="rated", aggfunc="max", fill_value=0)
    )
    save_table(reliability_matrix.reset_index(), "05_goal4", "reliability_four_ra_coverage_matrix")
    fig, ax = plt.subplots(figsize=(10, min(18, max(5, .08 * len(reliability_matrix)))))
    sns.heatmap(reliability_matrix, cmap=["#f2f2f2", "#59A14F"], cbar=False, ax=ax)
    ax.set(title="Crossed reliability coverage: four RAs per item", xlabel="Rater", ylabel="Recording × perceptual family")
    ax.tick_params(axis="y", labelleft=False)
    save_figure(fig, "05_goal4", "reliability_four_ra_rating_coverage")
    plt.show()


In [ ]:
# Reliability prevalence is shown beside agreement because imbalance can depress kappa.
if not consensus.empty and not agreement.empty:
    prevalence = (
        consensus.groupby("category")["consensus_rating"]
    .agg(
        n_consensus="count",
        positive=lambda x: int(pd.to_numeric(x, errors="coerce").sum()),
        prevalence=lambda x: float(pd.to_numeric(x, errors="coerce").mean()),
    ).reset_index()
    )
    agreement_view = agreement.merge(prevalence, on="category", how="left")
    save_table(agreement_view, "05_goal4", "reliability_agreement_and_prevalence")
    display(agreement_view)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.barplot(data=prevalence, x="prevalence", y="category", ax=axes[0], color="#59A14F")
    axes[0].set(title="Reliability-set 4RA consensus prevalence", xlabel="Positive fraction", ylabel="")
    axes[1].errorbar(
        agreement_view["gwet_ac1_nominal"],
        np.arange(len(agreement_view)),
        xerr=np.vstack([
            agreement_view["gwet_ac1_nominal"] - agreement_view["gwet_ac1_ci_low"],
            agreement_view["gwet_ac1_ci_high"] - agreement_view["gwet_ac1_nominal"],
        ]),
        fmt="o", color="0.2", ecolor="0.55", capsize=3,
    )
    axes[1].set_yticks(np.arange(len(agreement_view)), agreement_view["category"])
    axes[1].set(title="Complete-item Gwet AC1, bootstrap 95% CI", xlabel="Agreement", xlim=(-.1, 1.05))
    fig.tight_layout()
    save_figure(fig, "05_goal4", "reliability_prevalence_and_agreement")
    plt.show()
else:
    print("Reliability agreement is not estimable yet; inspect reliability_analysis_status.csv.")


In [ ]:
# Primary broad-coverage estimand: weighted within-rater effects in the distributed set.
main_alignment = read_stage("04_analysis/human_qc/main_distributed_rater_stratified_family_alignment")
main_effect = main_alignment.pivot(index="human_family", columns="objective_family", values="effect")
main_n = main_alignment.pivot(index="human_family", columns="objective_family", values="n_recordings")
save_table(main_alignment, "05_goal4", "main_rater_stratified_family_alignment")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(main_effect, vmin=-1, vmax=1, center=0, cmap="vlag", annot=True, fmt=".2f", ax=axes[0])
axes[0].set(title="Main-set rater-stratified effect", xlabel="Objective Q family", ylabel="Perceptual family")
sns.heatmap(main_n, cmap="viridis", annot=True, fmt=".0f", ax=axes[1])
axes[1].set(title="Pair-specific recording denominator", xlabel="Objective Q family", ylabel="")
fig.tight_layout()
save_figure(fig, "05_goal4", "main_rater_stratified_alignment_and_denominators")
plt.show()

matched_summary = (
    main_alignment.loc[main_alignment["estimable"]]
    .groupby("matched_family")["effect"]
    .agg(["count", "mean", "median"]).reset_index()
)
save_table(matched_summary, "05_goal4", "main_matched_vs_mismatched_descriptive")
display(matched_summary)

# Higher-confidence sensitivity: crossed-set consensus, if class support permits.
reliability_alignment = read_optional_stage("04_analysis/human_qc/reliability_four_ra_consensus_family_alignment")
if not reliability_alignment.empty:
    reliability_effect = reliability_alignment.pivot(index="human_family", columns="objective_family", values="effect")
    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(reliability_effect, vmin=-1, vmax=1, center=0, cmap="vlag", annot=True, fmt=".2f", ax=ax)
    ax.set(title="Reliability-subset 4RA consensus alignment", xlabel="Objective Q family", ylabel="Perceptual family")
    save_figure(fig, "05_goal4", "reliability_consensus_alignment")
    plt.show()


In [ ]:
# The merged 2RA workflow is comparable only for explicit shared families.
comparison = read_stage("04_analysis/human_qc/main_distributed_vs_two_ra_paired_alignment")
reliability_comparison = read_optional_stage("04_analysis/human_qc/reliability_four_ra_consensus_vs_two_ra_paired_alignment")
save_table(comparison, "05_goal4", "main_distributed_vs_two_ra_paired_alignment")
display(comparison)
if not reliability_comparison.empty:
    save_table(reliability_comparison, "05_goal4", "reliability_consensus_vs_two_ra_paired_alignment")
    display(reliability_comparison)

if "delta_auc_a_minus_b" in comparison and comparison["delta_auc_a_minus_b"].notna().any():
    plot = comparison.loc[comparison["status"].eq("ok")].copy()
    fig, ax = plt.subplots(figsize=(9, max(4, .8 * len(plot))))
    ax.errorbar(
        plot["delta_auc_a_minus_b"],
        np.arange(len(plot)),
        xerr=np.vstack([
            plot["delta_auc_a_minus_b"] - plot["delta_ci_low"],
            plot["delta_ci_high"] - plot["delta_auc_a_minus_b"],
        ]),
        fmt="o", capsize=3, color="0.2", ecolor="0.55",
    )
    ax.axvline(0, color="0.35", linestyle="--")
    ax.set_yticks(np.arange(len(plot)), plot["family"])
    ax.set(
        title="Paired shared-recording comparison: main distributed vs 2RA",
        xlabel="ΔAUC: distributed detailed − merged 2RA broad",
        ylabel="",
    )
    save_figure(fig, "05_goal4", "main_distributed_minus_two_ra_delta_auc")
    plt.show()


In [ ]:
# Secondary duration/fraction analysis preserves the richer interval annotations.
extent = read_stage("04_analysis/human_qc/main_distributed_extent_labels_secondary")
context = read_stage("04_analysis/human_qc/main_context_annotations_not_family_alignment")
reliability_extent = read_optional_stage("04_analysis/human_qc/reliability_four_ra_extent_consensus_secondary")
save_table(extent, "05_goal4", "main_distributed_extent_labels_secondary")

extent_summary = (
    extent.groupby("category")["annotated_fraction"]
    .agg(n="count", median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75))
    .reset_index()
)
save_table(extent_summary, "05_goal4", "main_extent_summary")
display(extent_summary)

if not reliability_extent.empty:
    reliability_extent_summary = (
        reliability_extent.groupby("category")["consensus_annotated_fraction"]
        .agg(n="count", median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75))
        .reset_index()
    )
    save_table(reliability_extent_summary, "05_goal4", "reliability_extent_consensus_summary")
    display(reliability_extent_summary)

context_summary = (
    context.groupby("category")["rating"]
    .agg(rater_recordings="size", positive="sum", prevalence="mean")
    .reset_index()
)
context_summary["primary_family_alignment"] = False
save_table(context_summary, "05_goal4", "context_annotations_excluded_from_family_alignment")
display(context_summary)


In [ ]:
goal4_reasons = []
if direction_audit.empty or not direction_audit["direction"].eq("higher_is_worse").all():
    goal4_reasons.append(
        "The 2RA scale direction is not confirmed as higher-is-worse; verify the codebook."
    )
if main_alignment.empty:
    goal4_reasons.append("Distributed detailed-rating family alignment is missing.")
if agreement.empty:
    goal4_reasons.append(
        "Complete four-RA crossed reliability agreement is missing or not estimable."
    )
goal4_ready = stage_gate(
    "Goal 4 perceptual family alignment",
    not goal4_reasons,
    goal4_reasons,
    "Open 06_results_registry_and_manuscript_tables.ipynb only after this gate passes.",
)